In [1]:
import pandas as pd
import fsspec
from zipfile import ZipFile
from io import BytesIO

def load_specific_csv_from_zip(url, filename):
    """Load one specific CSV from multi-file ZIP"""
    with fsspec.open(url, 'rb') as f:
        with ZipFile(BytesIO(f.read())) as z:
            return pd.read_csv(z.open(filename))

url = "gs://agntworks-data-dev/wheelsup/raw/AgntWorks.zip"

# Create quarter-wise DataFrames (exactly matching your file names)
# booked_flights = load_specific_csv_from_zip(url, 'AgntWorks/booked_flights.csv')
# requests_for_quotes = load_specific_csv_from_zip(url, 'AgntWorks/requests_for_quotes.csv')
# search_activity = load_specific_csv_from_zip(url, 'AgntWorks/search_activity.csv')
# search_results = load_specific_csv_from_zip(url, 'AgntWorks/search_results.csv')
# pricing_overrides = load_specific_csv_from_zip(url, 'AgntWorks/pricing_overrides.csv')


In [2]:
# print("="*50,"Booked Flights","="*50)
# display(booked_flights.head())
# print("="*50,"Request for Quotes","="*50)
# display(requests_for_quotes.sample(50))
# print("="*50,"Search Activity","="*50)
# display(search_activity.head())
# print("="*50,"Pricing Overrides","="*50)
# display(pricing_overrides.head())

In [3]:
# requests_for_quotes = load_specific_csv_from_zip(url, 'AgntWorks/requests_for_quotes.csv')
# display(requests_for_quotes.sample(10))

In [4]:
# pricing_overrides = load_specific_csv_from_zip(url, 'AgntWorks/pricing_overrides.csv')

# print("="*50,"Pricing Overrides","="*50)
# display(pricing_overrides.sample(10))

In [2]:
search_activity = load_specific_csv_from_zip(url, 'AgntWorks/search_activity.csv')
icao = "gs://agntworks-data-dev/sandbox/experiments/icao_cluster.csv"

df = search_activity.copy()
icao_map = pd.read_csv(icao)
cluster_dict = dict(zip(icao_map['icao'], icao_map['cluster']))

print(f'Loaded {len(df):,} searches')
print(f'Columns: {df.columns.tolist()}')
print(f'Sample routes:')
print(df['route'].head(5).tolist())

Loaded 765,682 searches
Columns: ['singleSearchRequestId', 'clientUserId', 'memberId', 'createDate', 'isCartAbandoned', 'requestingApp', 'route']
Sample routes:
['KMBO-KSAT-KMBO', 'KMBO-KSAT-KMBO', 'KDAL-KASE-KDAL', 'KLBE-KTEB-KBED-KLBE', 'LIMC-LEPA-LIMC']


In [3]:
search_activity.head(5)

,singleSearchRequestId,clientUserId,memberId,createDate,isCartAbandoned,requestingApp,route
0,7154439,946808,194306.0,2025-03-30T00:31:25.000Z,False,WU-Members iOS App,KMBO-KSAT-KMBO
1,7154443,946808,194306.0,2025-03-30T00:29:56.000Z,True,WU-Members iOS App,KMBO-KSAT-KMBO
2,7154446,70291,172664.0,2025-03-30T00:56:12.000Z,True,WU-Members Web Site,KDAL-KASE-KDAL
3,7154450,198321,258594.0,2025-03-30T02:35:59.000Z,True,WU-Members iOS App,KLBE-KTEB-KBED-KLBE
4,7154462,1218294,355103.0,2025-03-30T06:47:03.000Z,True,WU-Members iOS App,LIMC-LEPA-LIMC


In [4]:
# Split route into list, build consecutive (from, to) pairs
df['_airports'] = df['route'].str.split('-')
df['_legs']     = df['_airports'].apply(lambda x: list(zip(x[:-1], x[1:])))

# Explode: one row per leg
df_legs = df.explode('_legs').copy()
df_legs[['from_airport', 'to_airport']] = pd.DataFrame(
    df_legs['_legs'].tolist(), index=df_legs.index
)
df_legs = df_legs.drop(columns=['_airports', '_legs'])
df_legs = df_legs.reset_index(drop=True)

print(f'Original searches : {len(df):,}')
print(f'Expanded legs     : {len(df_legs):,}')
print(f'Avg legs/search   : {len(df_legs)/len(df):.2f}')

Original searches : 765,682
Expanded legs     : 1,183,154
Avg legs/search   : 1.55


In [5]:
df_legs['from_cluster'] = df_legs['from_airport'].map(cluster_dict).fillna('OTHER')
df_legs['to_cluster']   = df_legs['to_airport'].map(cluster_dict).fillna('OTHER')

unmapped_from = df_legs['from_cluster'].eq('OTHER').sum()
unmapped_to   = df_legs['to_cluster'].eq('OTHER').sum()
print(f'Unmapped from_airport: {unmapped_from:,} ({unmapped_from/len(df_legs)*100:.1f}%)')
print(f'Unmapped to_airport  : {unmapped_to:,} ({unmapped_to/len(df_legs)*100:.1f}%)')

# Time conversion — inline, no reload needed
df_legs['createDate_utc'] = pd.to_datetime(df_legs['createDate'], utc=True)
df_legs['createDate_et']  = df_legs['createDate_utc'].dt.tz_convert('US/Eastern').dt.tz_localize(None)
df_legs['date_et'] = df_legs['createDate_et'].dt.date
df_legs['hour_et'] = df_legs['createDate_et'].dt.hour
df_legs['dow_et']  = df_legs['createDate_et'].dt.day_name()
print(f'\nSample ET: {df_legs["createDate_et"].iloc[0]}')


Unmapped from_airport: 12,410 (1.0%)
Unmapped to_airport  : 13,051 (1.1%)

Sample ET: 2025-03-29 20:31:25


In [6]:
df_legs.head(1)

,singleSearchRequestId,clientUserId,memberId,createDate,isCartAbandoned,requestingApp,route,from_airport,to_airport,from_cluster,to_cluster,createDate_utc,createDate_et,date_et,hour_et,dow_et
0,7154439,946808,194306.0,2025-03-30T00:31:25.000Z,False,WU-Members iOS App,KMBO-KSAT-KMBO,KMBO,KSAT,OTHER_CLUSTER,HOUSTON_CLUSTER,2025-03-30 00:31:25+00:00,2025-03-29 20:31:25,2025-03-29,20,Saturday


In [7]:
# Sanity checks
print('=== Sample rows ===')
print(df_legs[['singleSearchRequestId','route','from_airport','to_airport','from_cluster','to_cluster']].head(10).to_string())

print('\n=== Top cluster pairs ===')
top_pairs = (
    df_legs.groupby(['from_cluster','to_cluster'])
    .size()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='searches')
)
print(top_pairs.to_string())

print('\n=== Route length distribution ===')
route_len = df['route'].str.split('-').apply(len)
print(route_len.value_counts().sort_index().rename('count').rename_axis('airports_in_route'))

=== Sample rows ===
   singleSearchRequestId                route from_airport to_airport           from_cluster             to_cluster
0                7154439       KMBO-KSAT-KMBO         KMBO       KSAT          OTHER_CLUSTER        HOUSTON_CLUSTER
1                7154439       KMBO-KSAT-KMBO         KSAT       KMBO        HOUSTON_CLUSTER          OTHER_CLUSTER
2                7154443       KMBO-KSAT-KMBO         KMBO       KSAT          OTHER_CLUSTER        HOUSTON_CLUSTER
3                7154443       KMBO-KSAT-KMBO         KSAT       KMBO        HOUSTON_CLUSTER          OTHER_CLUSTER
4                7154446       KDAL-KASE-KDAL         KDAL       KASE         DALLAS_CLUSTER          ASPEN_CLUSTER
5                7154446       KDAL-KASE-KDAL         KASE       KDAL          ASPEN_CLUSTER         DALLAS_CLUSTER
6                7154450  KLBE-KTEB-KBED-KLBE         KLBE       KTEB  WASHINGTON_DC_CLUSTER       NEW_YORK_CLUSTER
7                7154450  KLBE-KTEB-KBED-KLBE       

In [10]:
df_legs.to_csv('gs://agntworks-data-dev/wheelsup/raw/search_activity_route_split.csv', index=False)
print(f'Saved: gs://agntworks-data-dev/wheelsup/raw/search_activity_route_split.csv')
print(f'Rows : {len(df_legs):,}')
print(f'Cols : {df_legs.columns.tolist()}')


/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)


Saved: gs://agntworks-data-dev/wheelsup/raw/search_activity_route_split.csv
Rows : 1,183,154
Cols : ['singleSearchRequestId', 'clientUserId', 'memberId', 'createDate', 'isCartAbandoned', 'requestingApp', 'route', 'from_airport', 'to_airport', 'from_cluster', 'to_cluster', 'createDate_utc', 'createDate_et', 'date_et', 'hour_et', 'dow_et']


In [8]:
pd.read_csv('gs://agntworks-data-dev/wheelsup/processed/wingx/lj_smid_us_revenue_clean_24_25_metro_metadata.csv')

,Filter Category,Applied Criteria,Action Taken
0,Geography,Strictly US-Domestic (Both FromAirport and ToA...,Excluded International & Non-K airports
1,Aircraft Segments,"Light Jet, Super Midsize Jet","Excluded Heavy, Midsize, Turboprop, etc."
2,Minimum Flight Duration,Hours >= 0.5 (Revenue Missions),Excluded short hops and positioning legs
3,Mapping Source,transformed_icao_metro.csv (98.3% coverage),Enriched with Metro Area names
4,Data Period,Full Year 2024 through Full Year 2025,Consolidated from multiple ZIP archives
